# Hierarchischer Ansatz: Weiches Routing

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Prüfen, ob weiches Routing über P(Kultur) · P(Stadium | Kultur) die Fehlerfortpflanzung der harten Kulturentscheidung senkt (#88).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, clone
from sklearn.pipeline import Pipeline

from awp2.config import (
    CROP_COL,
    HARD_ROUTING,
    MODEL_DOCS_FIGURES_DIR,
    SOFT_ROUTING,
    STAGE_COL,
)
from awp2.data import load_folds, load_split
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models import make_hierarchical_svm
from awp2.models.tuned import tuned_hierarchical_svm
from awp2.plots import plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig, build_preprocessor

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

In `hierarchical_svm` (#87) kosten Kulturfehler rund 0.095 BAcc kombiniert (Oracle-Kultur 0.939
statt 0.844). Beim harten Routing entscheidet allein die Kultur-SVM, welches Stadium-Modell
antwortet. Beim weichen Routing gewinnt das Paar mit dem höchsten P(Kultur) · P(Stadium | Kultur) –
ein sicheres Stadium-Modell kann eine unsichere Kulturentscheidung überstimmen.

Gegenhypothese: Jedes Stadium-Modell kennt nur Spektren seiner Kultur; für fremde Spektren sind
seine Wahrscheinlichkeiten bedeutungslos, aber oft trotzdem hoch. Kulturen mit wenigen Stadien
(Reis: 2) haben außerdem von vornherein höhere P(Stadium | Kultur).

Vorgehen: Die SVMs bekommen Wahrscheinlichkeiten über `CalibratedClassifierCV` (Sigmoid,
`ensemble=False`). Weiches Routing wird per CV getunt und gegen den CV-Score der harten Suche aus
#87 verglichen (gleiche Folds, gleiche Vorverarbeitung); **nur die CV entscheidet**. Hinweis: Ein
Smoke-Test hatte weiches Routing (damals mit `SVC(probability=True)`) einmal auf der Validierung
gesehen (0.823); das fließt nicht in die Entscheidung ein.

In [ ]:
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
param_grid = {
    "crop_estimator__estimator__C": [10.0, 30.0, 100.0],
    "crop_estimator__estimator__gamma": [0.003, 0.01, 0.03],
    "stage_estimator__estimator__C": [30.0, 100.0],
    "stage_estimator__estimator__gamma": [0.003],
}
reference = tuned_hierarchical_svm()
hard_cv_score = reference.cv_score
reference.tuning_run, hard_cv_score

## CV-Suche: weiches Routing

In [ ]:
tuned = tune(
    make_hierarchical_svm(SOFT_ROUTING),
    TuneConfig(
        name="hierarchical_soft_search",
        approach="hierarchical",
        description="RBF-SVM → RBF-SVM mit kalibrierten Wahrscheinlichkeiten "
        "(CalibratedClassifierCV) und weichem Routing; Kultur-C/gamma und Stadium-C um das "
        "Optimum aus #87. Vergleich gegen die harte Suche hierarchical_svm_search per CV.",
        preprocessing=preprocessing,
        param_grid=param_grid,
    ),
)
cv_comparison = pd.Series(
    {
        f"hart (hierarchical_svm_search, {reference.tuning_run})": hard_cv_score,
        "weich (hierarchical_soft_search)": tuned.best_score,
    },
    name="CV BAcc kombiniert",
)
tuned.best_params, cv_comparison.round(4)

In [ ]:
tuning_ax = plot_tuning_candidates(tuned.candidates, color_by="crop_estimator__estimator__C")
top_row = len(tuned.candidates) - 0.4
tuning_ax.axvline(hard_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.text(hard_cv_score, top_row, " hart (#87)", fontsize="small", va="top")
tuning_ax.set_xlim(0.75, 0.87)
tuning_ax.get_legend().set_loc("lower left")
tuning_ax.set_title("Weiches Routing: CV-Kandidaten gegen das beste harte Routing")
save_doc_figure(tuning_ax.figure, "hierarchical_soft_tuning", MODEL_DOCS_FIGURES_DIR)

## Diagnose auf den Trainings-Folds

Was ändert weiches Routing? Out-of-fold-Vorhersagen dreier Varianten mit den besten
Hyperparametern – nur Trainingsdaten, keine Validierung. „hart (kalibriert)“ trennt den Effekt
der Kalibrierung (argmax der Wahrscheinlichkeiten statt SVC-Stimmen) vom Effekt des Routings.

In [ ]:
def fold_pipeline(model: BaseEstimator, fold_train: np.ndarray) -> Pipeline:
    pipeline = Pipeline(
        [("preprocess", build_preprocessor(preprocessing)), ("model", clone(model))]
    )
    return pipeline.fit(split.X_train.iloc[fold_train], split.y_train.iloc[fold_train])


variants = {
    "hart (SVC)": reference.model,
    "hart (kalibriert)": clone(tuned.best_model).set_params(routing=HARD_ROUTING),
    "weich (kalibriert)": tuned.best_model,
}
oof_config = RunConfig(
    name="hierarchical_soft_oof",
    approach="hierarchical",
    preprocessing=reference.preprocessing,
    track=False,
)
oof = {label: out_of_fold_predictions(model, oof_config) for label, model in variants.items()}

In [ ]:
true_crop = split.y_train[CROP_COL]
crop_recall = pd.DataFrame(
    {label: (pred[CROP_COL] == true_crop).groupby(true_crop).mean() for label, pred in oof.items()}
)


def changes(first: str, second: str) -> pd.Series:
    crop_changed = oof[first][CROP_COL] != oof[second][CROP_COL]
    stage_changed = ~crop_changed & (oof[first][STAGE_COL] != oof[second][STAGE_COL])
    return pd.Series(
        {"Kultur geändert": crop_changed.sum(), "nur Stadium geändert": stage_changed.sum()},
        name=f"{first} → {second}",
    )


change_counts = pd.DataFrame(
    [changes("hart (SVC)", "hart (kalibriert)"), changes("hart (kalibriert)", "weich (kalibriert)")]
)
crop_recall.round(3), change_counts, len(true_crop)

In [ ]:
def crop_probability_ratio(model: BaseEstimator) -> pd.DataFrame:
    rows = []
    for fold in load_folds():
        pipeline = fold_pipeline(model, fold.train)
        features = pipeline[:-1].transform(split.X_train.iloc[fold.val])
        crop_model = pipeline[-1].crop_estimator_
        proba = np.sort(crop_model.predict_proba(features), axis=1)
        rows.append(
            pd.DataFrame(
                {
                    "ratio": proba[:, -1] / proba[:, -2],
                    "correct": crop_model.predict(features)
                    == split.y_train[CROP_COL].iloc[fold.val].to_numpy(),
                }
            )
        )
    return pd.concat(rows, ignore_index=True)


RATIO_THRESHOLD = 2.0
ratios = crop_probability_ratio(tuned.best_model)
ratios.groupby("correct")["ratio"].agg(
    median="median", share_above=lambda r: (r > RATIO_THRESHOLD).mean(), n="size"
).round(3)

**Befund:** Die Kalibrierung allein ändert keine Kulturentscheidung (nur 5 Stadien), weiches
Routing danach nur 2 von 3 912 Kulturentscheidungen; der Recall je Kultur ist in allen drei
Varianten gleich. Grund: Auch bei *falschen* Kulturentscheidungen ist die Top-Kultur im Median
3,3-mal so wahrscheinlich wie die zweite (in 90 % > 2-mal) – kaum weniger als bei richtigen
(3,4; 95 %). Die kalibrierten Wahrscheinlichkeiten erben die Stimmenstruktur der OvO-SVM; ein
Stadium-Modell kann eine Kultur damit fast nie überstimmen. Das zeigt: weiches Routing hilft
*mit diesen Wahrscheinlichkeiten* nicht – nicht, dass es grundsätzlich nicht helfen kann.
Reis wird nicht bevorzugt; die Gegenhypothese „wenige Stadien“ trifft nicht zu.

## Einmalige Bewertung

Der CV-Gewinner wird einmal mit `run()` bewertet, wenn weiches Routing in der CV vorn liegt;
sonst bleibt `hierarchical_svm` (#87) die Referenz. Ein Vorsprung im Rauschen bleibt ein
Unentschieden – der Lauf dient nur der vollständigen Dokumentation.

In [ ]:
if tuned.best_score > hard_cv_score:
    result = run(
        tuned.best_model,
        RunConfig(
            name="hierarchical_soft",
            approach="hierarchical",
            description="Bestes weiches Routing aus hierarchical_soft_search, einmal bewertet.",
            preprocessing=preprocessing,
            tuning_run=tuned.run_id,
        ),
    )
    summary = result.metrics.model_dump()
else:
    summary = {"Entscheidung": "hart bleibt; Referenz hierarchical_svm (#87)"}
summary, tuned.run_id